In [1]:
import altair as alt 
import pandas as pd
import numpy as np

import sys
sys.path.insert(0, '..')

from dataloader import load_df, load_sensitivity, PARAMS, load_sensitivity_perframe, load_perframe_pervalue, PARAM_LABELS

In [2]:
df          = load_df("../dataset.json")
sensitivity = load_sensitivity("../dataset.json")
sensitivity_perframe = load_sensitivity_perframe("../dataset.json")

/Users/mariasilva/Documents/PerceptualTAA/video_parameterization/../dataloader.py:103: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_center_group)
/Users/mariasilva/Documents/PerceptualTAA/video_parameterization/../dataloader.py:103: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_center_group)
/Users/mariasilva/Documents/PerceptualTAA/video_parameterization/../dataloader.py:224: FutureWarning: DataFram

### Main Plot (one regression line)

In [3]:
import altair as alt
import pandas as pd
from scipy import stats
from dataloader import load_sensitivity

# ── Load data ─────────────────────────────────────────────────────────────────
features = pd.read_csv('video_stats.csv')
features['scene'] = features['filename'].str.replace('-demo.mp4', '').str.replace('.mp4', '')
features = features.drop(columns=['filename', 'error'])
features = features.rename(columns={
    'SI':  'spatial_info',
    'TI':  'temporal_info',
    'CF':  'colorfulness',
    'TP':  'texture',
    'MV':  'motion',
    'DTP': 'dynamic_texture',
})
features['scene'] = features['scene'].replace('fantasticvillage', 'fantasticvillage-open')
sensitivity = load_sensitivity('../dataset.json')

df = sensitivity.merge(features, on='scene')
print(df.groupby('parameter')['sensitivity_std'].agg(['mean', 'std', 'max']).round(3))


FEATURE_COLS = ['spatial_info', 'temporal_info', 'colorfulness', 'texture', 'motion', 'dynamic_texture']
PARAM_LABELS = {
    'alpha_weight': 'Alpha weight',
    'hist_percent': 'History %',
    'filter_size':  'Filter size',
    'num_samples':  'Num samples',
}

# FEATURE_COLS_LABELS = 


df_scatter = df.copy()
df_scatter['parameter'] = df_scatter['parameter'].map(PARAM_LABELS)

df_melt = df_scatter.melt(
    id_vars=['scene', 'resolution', 'parameter', 'sensitivity_std', 'sensitivity_range'],
    value_vars=FEATURE_COLS,
    var_name='feature',
    value_name='feature_value',
)

df_melt['resolution'] = df_melt['resolution'].astype(str) + '%'
res_order  = ['100%', '87%', '71%', '50%']
res_colors = ['#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']
color_scale = alt.Scale(domain=res_order, range=res_colors)

# ── Pearson correlations per (feature × parameter) cell ───────────────────────
corr_rows = []
for (feat, param), grp in df_melt.groupby(['feature', 'parameter']):
    clean = grp.groupby('scene')[['feature_value', 'sensitivity_std']].mean().dropna()
    if len(clean) > 2:
        r, p = stats.pearsonr(clean['feature_value'], clean['sensitivity_std'])
    else:
        r, p = float('nan'), float('nan')
    corr_rows.append({'feature': feat, 'parameter': param, 'pearson_r': r, 'p_value': p})

df_corr = pd.DataFrame(corr_rows)
df_corr['label'] = df_corr.apply(
    lambda row: f"r = {row['pearson_r']:+.2f}",
    axis=1,
)

# Top-right anchor: use per-feature x max
x_maxs = df_melt.groupby('feature')['feature_value'].max().rename('x_anchor')
df_corr = df_corr.join(x_maxs, on='feature')

# ── Join back so both layers share one dataframe ──────────────────────────────
df_melt = df_melt.merge(df_corr[['feature', 'parameter', 'label', 'x_anchor']],
                         on=['feature', 'parameter'], how='left')

base = alt.Chart(df_melt)

scatter_layer = (
    base
    .mark_circle(size=55, opacity=0.65)
    .encode(
        x=alt.X('feature_value:Q', title=None, scale=alt.Scale(zero=False)),
        y=alt.Y('sensitivity_std:Q', title='Sensitivity (std)'),
        color=alt.Color('resolution:N', title='Resolution',
                        scale=color_scale, sort=res_order),
        tooltip=['scene', 'parameter', 'resolution', 'feature',
                 alt.Tooltip('feature_value:Q', format='.2f'),
                 alt.Tooltip('sensitivity_std:Q', format='.3f')],
    )
)

corr_layer = (
    base
    .mark_text(align='right', baseline='top', fontSize=9, color='#333333', fontStyle='italic')
    .encode(
        x=alt.X('x_anchor:Q', scale=alt.Scale(zero=False)),
        y=alt.value(4),
        text=alt.Text('label:N'),
    )
)

regression_layer = (
    base
    .mark_line(opacity=0.5, strokeWidth=1.5, color='#333333')
    .transform_regression('feature_value', 'sensitivity_std')
    .encode(
        x=alt.X('feature_value:Q'),
        y=alt.Y('sensitivity_std:Q'),
    )
)

scatter = (
    (scatter_layer + regression_layer + corr_layer)
    .properties(width=160, height=130)
    .facet(
        row=alt.Row('parameter:N', sort=list(PARAM_LABELS.values()),
                    header=alt.Header(labelAngle=0, labelAlign='left', labelLimit=200)),
        column=alt.Column('feature:N', sort=FEATURE_COLS),
    )
    .resolve_scale(x='independent')
)

scatter

import vl_convert as vlc

vg_json = scatter.to_dict()
png_data = vlc.vegalite_to_png(vg_json, scale=3)
with open('sensitivity_plot.png', 'wb') as f:
    f.write(png_data)

/Users/mariasilva/Documents/PerceptualTAA/video_parameterization/../dataloader.py:103: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_center_group)


               mean    std    max
parameter                        
alpha_weight  1.323  1.510  8.530
filter_size   0.065  0.108  0.502
hist_percent  0.622  0.506  1.838
num_samples   0.089  0.117  0.581


In [ ]:
import altair as alt
import pandas as pd
from scipy import stats
from dataloader import load_sensitivity, RESOLUTION_LABELS, RESOLUTIONS

# ── Load data ─────────────────────────────────────────────────────────────────
features = pd.read_csv('video_stats.csv')
features['scene'] = features['filename'].str.replace('-demo.mp4', '').str.replace('.mp4', '')
features = features.drop(columns=['filename', 'error'])
features = features.rename(columns={
    'SI':  'spatial_info',
    'TI':  'temporal_info',
    'CF':  'colorfulness',
    'TP':  'texture',
    'MV':  'motion',
    'DTP': 'dynamic_texture',
})
features['scene'] = features['scene'].replace('fantasticvillage', 'fantasticvillage-open')
sensitivity = load_sensitivity('../dataset.json')

df = sensitivity.merge(features, on='scene')
print(df.groupby('parameter')['sensitivity_std'].agg(['mean', 'std', 'max']).round(3))


FEATURE_COLS = ['spatial_info', 'temporal_info', 'colorfulness', 'texture', 'motion', 'dynamic_texture']
PARAM_LABELS = {
    'alpha_weight': 'Alpha weight',
    'hist_percent': 'History %',
    'filter_size':  'Filter size',
    'num_samples':  'Num samples',
}

df_scatter = df.copy()
df_scatter['parameter'] = df_scatter['parameter'].map(PARAM_LABELS)

df_melt = df_scatter.melt(
    id_vars=['scene', 'resolution', 'parameter', 'sensitivity_std', 'sensitivity_range'],
    value_vars=FEATURE_COLS,
    var_name='feature',
    value_name='feature_value',
)

df_melt['resolution'] = df_melt['resolution'].astype(int).map(RESOLUTION_LABELS)

res_order  = [RESOLUTION_LABELS[k] for k in [100, 87, 71, 50]]  # ['100%', '75%', '50%', '25%']
res_colors = ['#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']
color_scale = alt.Scale(domain=res_order, range=res_colors)


# res_colors = ['#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']
# color_scale = alt.Scale(domain=RESOLUTIONS, range=res_colors)

# ── Pearson correlations per (feature × parameter) cell ───────────────────────
corr_rows = []
for (feat, param), grp in df_melt.groupby(['feature', 'parameter']):
    clean = grp.groupby('scene')[['feature_value', 'sensitivity_std']].mean().dropna()
    if len(clean) > 2:
        r, p = stats.pearsonr(clean['feature_value'], clean['sensitivity_std'])
    else:
        r, p = float('nan'), float('nan')
    corr_rows.append({'feature': feat, 'parameter': param, 'pearson_r': r, 'p_value': p})

df_corr = pd.DataFrame(corr_rows)
df_corr['label'] = df_corr.apply(
    lambda row: f"r = {row['pearson_r']:+.2f}",
    axis=1,
)


corr_rows_res = []
for (feat, param, res), grp in df_melt.groupby(['feature', 'parameter', 'resolution']):
    clean = grp[['feature_value', 'sensitivity_std']].dropna()
    if len(clean) > 2:
        r, p = stats.pearsonr(clean['feature_value'], clean['sensitivity_std'])
    else:
        r, p = float('nan'), float('nan')
    corr_rows_res.append({'feature': feat, 'parameter': param, 'resolution': res, 'pearson_r': r, 'p_value': p})

df_corr_res = pd.DataFrame(corr_rows_res)
df_corr_res['label_res'] = df_corr_res['pearson_r'].apply(lambda r: f"r = {r:+.2f}")

# anchor each resolution label at its own x_max
x_maxs_res = df_melt.groupby(['feature', 'resolution'])['feature_value'].max().rename('x_anchor_res')
df_corr_res = df_corr_res.join(x_maxs_res, on=['feature', 'resolution'])

df_melt = df_melt.merge(
    df_corr_res[['feature', 'parameter', 'resolution', 'label_res', 'x_anchor_res']],
    on=['feature', 'parameter', 'resolution'], how='left'
)

# Top-right anchor: use per-feature x max
x_maxs = df_melt.groupby('feature')['feature_value'].max().rename('x_anchor')
df_corr = df_corr.join(x_maxs, on='feature')

# ── Join back so both layers share one dataframe ──────────────────────────────
df_melt = df_melt.merge(df_corr[['feature', 'parameter', 'label', 'x_anchor']],
                         on=['feature', 'parameter'], how='left')

base = alt.Chart(df_melt)

scatter_layer = (
    base
    .mark_circle(size=35, opacity=0.65)
    .encode(
        x=alt.X('feature_value:Q', title=None, scale=alt.Scale(zero=False)),
        y=alt.Y('sensitivity_std:Q', title='Sensitivity (std)'),
        color=alt.Color('resolution:N', title='Resolution',
                        scale=color_scale, sort=res_order),
        tooltip=['scene', 'parameter', 'resolution', 'feature',
                 alt.Tooltip('feature_value:Q', format='.2f'),
                 alt.Tooltip('sensitivity_std:Q', format='.3f')],
    )
)

corr_layer = (
    base
    .mark_text(align='right', baseline='top', fontSize= 10, color='#333333', fontStyle='italic')
    .encode(
        x=alt.X('x_anchor:Q', scale=alt.Scale(zero=False)),
        y=alt.value(3),
        text=alt.Text('label:N'),
    )
)

corr_res_layer = (
    base
    .mark_text(align='right', baseline='top', fontSize=10, fontStyle='italic', fontWeight='bold')
    .encode(
        x=alt.X('x_anchor_res:Q', scale=alt.Scale(zero=False)),
        y=alt.value(3),   # will stack via offset below
        text=alt.Text('label_res:N'),
        color=alt.Color('resolution:N', title='Resolution',
                        scale=color_scale, sort=res_order),
        yOffset=alt.YOffset('resolution:N',
                             sort=res_order,
                             scale=alt.Scale(range=[10, 58])),  # stack 4 labels below global
    )
    .transform_aggregate(
        label_res='min(label_res)',
        x_anchor_res='max(x_anchor_res)',
        groupby=['feature', 'parameter', 'resolution']
    )
)

regression_layer = (
    base
    .mark_line(opacity=0.5, strokeWidth=1.5, clip=True)
    .transform_regression(
        'feature_value', 'sensitivity_std',
        groupby=['resolution'],
    )
    .encode(
        x=alt.X('feature_value:Q'),
        y=alt.Y('sensitivity_std:Q'),
        color=alt.Color('resolution:N', title='Resolution',
                        scale=color_scale, sort=res_order),
    )
)

scatter = (
    (scatter_layer + regression_layer + corr_layer + corr_res_layer)
    .properties(width=160, height=130)
    .facet(
        row=alt.Row('parameter:N', sort=list(PARAM_LABELS.values()),
                    header=alt.Header(labelAngle=0, labelAlign='left', labelLimit=200)),
        column=alt.Column('feature:N', sort=FEATURE_COLS),
    )
    .resolve_scale(x='independent')
)
scatter

# import vl_convert as vlc

# vg_json = scatter.to_dict()
# png_data = vlc.vegalite_to_png(vg_json, scale=3)
# with open('sensitivity_plot_indiv-lines.png', 'wb') as f:
#     f.write(png_data)

/Users/mariasilva/Documents/PerceptualTAA/video_parameterization/../dataloader.py:103: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_center_group)


               mean    std    max
parameter                        
alpha_weight  1.323  1.510  8.530
filter_size   0.065  0.108  0.502
hist_percent  0.622  0.506  1.838
num_samples   0.089  0.117  0.581


In [11]:
print("unique resolutions in df_melt:", df_melt['resolution'].unique())
print("res_order:", res_order)
print("color_scale domain:", color_scale.domain)
print("df_melt shape:", df_melt.shape)
print(df_melt[['scene', 'resolution', 'feature', 'sensitivity_std', 'feature_value']].head(10))

unique resolutions in df_melt: ['25%' '50%' '75%' '100%']
res_order: ['100%', '75%', '50%', '25%']
color_scale domain: ['100', '87', '71', '50']
df_melt shape: (1920, 11)
       scene resolution       feature  sensitivity_std  feature_value
0  abandoned        25%  spatial_info         1.393165       0.071411
1  abandoned        25%  spatial_info         1.393165       0.073039
2  abandoned        25%  spatial_info         0.040411       0.071411
3  abandoned        25%  spatial_info         0.040411       0.073039
4  abandoned        25%  spatial_info         0.265683       0.071411
5  abandoned        25%  spatial_info         0.265683       0.073039
6  abandoned        25%  spatial_info         0.036306       0.071411
7  abandoned        25%  spatial_info         0.036306       0.073039
8  abandoned        50%  spatial_info         0.761244       0.071411
9  abandoned        50%  spatial_info         0.761244       0.073039
